In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

In [ ]:
class FraudFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Personaliced Transformer to apply Feature Engineering to transaction data before feeding it to the classification 
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        
        if 'transaction_velocity_24h' in X_copy.columns and 'failed_transactions_24h' in X_copy.columns:
            X_copy['failed_velocity_ratio'] = X_copy['failed_transactions_24h'] / (X_copy['transaction_velocity_24h'] + 1)
            X_copy['velocity_failure_index'] = X_copy['transaction_velocity_24h'] * X_copy['failed_transactions_24h']
            X_copy['brute_force_warning'] = np.where(
                (X_copy['transaction_velocity_24h'] >= 3) & (X_copy['failed_transactions_24h'] >= 2), 1, 0
            )
            
        return X_copy

In [ ]:
print("Loading data...")
df = pd.read_csv('../data/fraud_dataset.csv')

df_clean = df.drop(columns=['transaction_id', 'customer_id'])

X = df_clean.drop(columns=['fraud'])
y = df_clean['fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols.extend(['failed_velocity_ratio', 'velocity_failure_index', 'brute_force_warning'])

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

In [ ]:
pipeline = Pipeline(steps=[
    ('feature_engineer', FraudFeatureEngineer()),
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

In [ ]:
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [None, 5, 10],
    'classifier__min_samples_split': [2, 5]
}

print("Initializing GridSearchCV (optimizing by Recall)...")
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='recall',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print(f"\n--- Best Hyperparameters ---\n{grid_search.best_params_}")

In [ ]:
y_pred = best_model.predict(X_test)

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

In [ ]:
os.makedirs('../app/services/models', exist_ok=True)
model_route = '../app/services/models/fraud_pipeline.joblib'

joblib.dump(best_model, model_route)
print(f"\nModel exported to: {model_route}")